# XTracker — Post-Counter Markets

Polymarket's **XTracker** service at [xtracker.polymarket.com](https://xtracker.polymarket.com)
tracks posts by public figures (Elon Musk, Trump, Zelenskyy, etc.) to power the
"how many tweets/posts in a window" counter markets.

All 7 endpoints are **public** — no authentication needed:

| Method | Description |
|---|---|
| `get_xtracker_users` | List all tracked users |
| `get_xtracker_user` | Single user detail (with trackings) |
| `get_xtracker_user_posts` | Posts by a user in a date range |
| `get_xtracker_user_trackings` | Tracking periods for a user |
| `get_xtracker_trackings` | All tracking periods across users |
| `get_xtracker_tracking` | Single tracking with optional stats |
| `get_xtracker_metrics` | Daily post metrics for a user |

In [ ]:
!pip install -q polymarket-pandas

In [ ]:
import pandas as pd

from polymarket_pandas import PolymarketPandas

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

client = PolymarketPandas()

## Tracked Users

`get_xtracker_users` returns all users the xtracker service monitors.
With `stats=True` each row includes aggregate counts; `expand_count=True`
(the default) flattens the `_count` dict into a `countPosts` column.

In [ ]:
users = client.get_xtracker_users(stats=True, expand_count=True)

print(f"Tracked users: {len(users)}")
print(f"Columns: {list(users.columns)}")
print()
users

## Single User Detail

`get_xtracker_user` returns a dict for one user. The `trackings` field
is automatically converted into a DataFrame showing each tracking period
(i.e. each counter market window) configured for this user.

In [ ]:
# Pick the first handle from the users DataFrame
handle = users["handle"].iloc[0]
print(f"Looking up user: {handle}")
print()

user = client.get_xtracker_user(handle=handle)

# Show scalar fields
for key, value in user.items():
    if key != "trackings":
        print(f"  {key}: {value}")

# Show the trackings DataFrame
print(f"\nTrackings ({len(user['trackings'])} periods):")
user["trackings"]

## User Posts (Last 7 Days)

`get_xtracker_user_posts` fetches actual tracked posts for a user within
a date range. Date parameters accept `pd.Timestamp` objects.

In [ ]:
today = pd.Timestamp.now(tz="UTC").normalize()
start = today - pd.Timedelta(days=7)

print(f"Fetching posts for @{handle} from {start.date()} to {today.date()}")

posts = client.get_xtracker_user_posts(
    handle=handle,
    start_date=start,
    end_date=today,
)

print(f"Posts found: {len(posts)}")
print(f"Columns: {list(posts.columns)}")
print()
posts.head(10)

## Active Trackings with User Info

`get_xtracker_trackings` lists all tracking periods across all users.
Each tracking represents one counter market window. With `expand_user=True`,
the nested user dict is flattened into prefixed columns (`user_handle`,
`user_platform`, etc.).

In [ ]:
trackings = client.get_xtracker_trackings(active_only=True, expand_user=True)

print(f"Active trackings: {len(trackings)}")
print(f"Columns: {list(trackings.columns)}")
print()
trackings.head(10)

## Single Tracking with Stats

`get_xtracker_tracking` fetches a single tracking period by ID.
With `include_stats=True`, the `stats` field becomes a DataFrame of daily
buckets. Aggregate scalars (total, pace, percentComplete, etc.) are stored
on `stats.attrs` so nothing is lost.

In [ ]:
# Pick a tracking ID from the active trackings
tracking_id = trackings["id"].iloc[0]
print(f"Tracking ID: {tracking_id}")

tracking = client.get_xtracker_tracking(id=tracking_id, include_stats=True)

# Show scalar fields
for key, value in tracking.items():
    if key not in ("stats", "user"):
        print(f"  {key}: {value}")

# Show aggregate stats from attrs
print("\nAggregate stats:")
for k, v in tracking["stats"].attrs.items():
    print(f"  {k}: {v}")

# Show daily breakdown
print(f"\nDaily buckets ({len(tracking['stats'])} rows):")
tracking["stats"]

## Daily Metrics

`get_xtracker_metrics` returns per-bucket post metrics for a user over
a date range. The nested `data` object on each metric is flattened into
prefixed columns (`dataCount`, `dataCumulative`, `dataTrackingId`).

In [ ]:
# Use the internal user ID from the user dict we fetched earlier
user_id = user["id"]
print(f"User ID: {user_id}  (handle: {handle})")
print(f"Date range: {start.date()} to {today.date()}")
print()

metrics = client.get_xtracker_metrics(
    user_id=user_id,
    start_date=start,
    end_date=today,
)

print(f"Metric rows: {len(metrics)}")
print(f"Columns: {list(metrics.columns)}")
print()
metrics